# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
pd.set_option('display.max_colwidth', None)

# PNAD Income MetaData 

This project builds a **consistent, longitudinal dataset of household income in Brazil** using microdata from the *Pesquisa Nacional por Amostra de Domicílios (PNAD)* and *PNAD Contínua*. The main challenge is structural heterogeneity across years:

- Variable names change (`V2954`, `V4614`, `V4621`, `VD4019`, etc.)
- Fixed-width positions and lengths vary
- Some years include household size, others do not
- Some years have **no survey**
- Post-2016 data comes from **PNAD Contínua (quarterly)**

To solve this, a unified specification table (`df_specs`) was created with:

- `ano`: reference year  
- `var_renda`: income variable  
- `pos_renda`, `tam_renda`: extraction specs  
- `var_morador`: household size (when available)  
- `missing_renda`: missing-income sentinel code  
- `raw_subdir`, `raw_pattern`, `n_files`: raw-file ingestion specification  
- `link`: official IBGE source  

This enables:

- Automated ingestion (`read_fwf`)
- Standardization to a single variable (`renda`)
- Per capita income computation
- Inflation and FX normalization
- Distribution analysis (CCDF, log-log, double-log)

The final dataset is:

- **Time-consistent (1976–2025)**
- **Economically comparable**
- Suitable for **econophysics analysis** (heavy tails, Pareto, etc.)

---

## Data Sources (IBGE FTP)

1. 1976 — [PNAD Year 1976](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1976/)  
2. 1977 — [PNAD Year 1977](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1977/)  
3. 1978 — [PNAD Year 1978](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1978/)  
4. 1979 — [PNAD Year 1979](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1979/)  
5. 1980 — No PNAD survey  
6. 1981 — [PNAD Year 1981](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1981/)  
7. 1982 — [PNAD Year 1982](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1982/)  
8. 1983 — [PNAD Year 1983](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1983/)  
9. 1984 — [PNAD Year 1984](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1984/)  
10. 1985 — [PNAD Year 1985](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1985/)  
11. 1986 — [PNAD Year 1986](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1986/)  
12. 1987 — [PNAD Year 1987](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1987/)  
13. 1988 — [PNAD Year 1988](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1988/)  
14. 1989 — [PNAD Year 1989](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1989/)  
15. 1990 — [PNAD Year 1990](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1990/PND1990N.DAT)  
16. 1991 — No PNAD survey  
17. 1992 — [PNAD Year 1992](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1992/)  
18. 1993 — [PNAD Year 1993](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1993/)  
19. 1994 — No PNAD survey  
20. 1995 — [PNAD Year 1995](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1995/)  
21. 1996 — [PNAD Year 1996](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1996/)  
22. 1997 — [PNAD Year 1997](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1997/)  
23. 1998 — [PNAD Year 1998](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1998/)  
24. 1999 — [PNAD Year 1999](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1999/)  
25. 2000 — No PNAD survey  
26. 2001 — [PNAD Year 2001](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2001.zip)  
27. 2002 — [PNAD Year 2002](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2002.zip)  
28. 2003 — [PNAD Year 2003](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2003_20150814.zip)  
29. 2004 — [PNAD Year 2004](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2004.zip)  
30. 2005 — [PNAD Year 2005](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2005.zip)  
31. 2006 — [PNAD Year 2006](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2006.zip)  
32. 2007 — [PNAD Year 2007](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2007_20150814.zip)  
33. 2008 — [PNAD Year 2008](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2008.zip)  
34. 2009 — [PNAD Year 2009](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2009_20171228.zip)  
35. 2010 — No PNAD survey  
36. 2011 — [PNAD Year 2011](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2011_20150814.zip)  
37. 2012 — [PNAD Year 2012](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2012_20150814.zip)  
38. 2013 — [PNAD Year 2013](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/2013/Dados_20170807.zip)  
39. 2014 — [PNAD Year 2014](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/2014/Dados_20170323.zip)  
40. 2015 — [PNAD Year 2015](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/2015/Dados_20170517.zip)  
41. 2016 — [PNAD Year 2016](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2016/)  
42. 2017 — [PNAD Year 2017](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2017/)  
43. 2018 — [PNAD Year 2018](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2018/)  
44. 2019 — [PNAD Year 2019](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2019/)  
45. 2020 — [PNAD Year 2020](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2020/)  
46. 2021 — [PNAD Year 2021](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2021/)  
47. 2022 — [PNAD Year 2022](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2022/)  
48. 2023 — [PNAD Year 2023](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2023/)  
49. 2024 — [PNAD Year 2024](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2024/)  
50. 2025 — [PNAD Year 2025](https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2025/)  

In [2]:
# define função para construir o dataframe de especificações
def build_specs_pnad_df(): 
    # dicionário com metadados por ano (variáveis, posições e links)
    specs_pnad = {  
        1976: ('V2954', 227, 9, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1976/'),
        1977: ('V131', 288, 9, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1977/'),
        1978: ('V2541', 214, 9, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1978/'),
        1979: ('V2517', 167, 9, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1979/'),
        1980: (None, None, None, None, None, None, None),
        1981: ('V5010', 223, 7, 'V9329', 219, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1981/'),
        1982: ('V602', 199, 7, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1982/'),
        1983: ('V5010', 223, 7, 'V9329', 219, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1983/'),
        1984: ('V5010', 223, 7, 'V9329', 219, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1984/'),
        1985: ('V5010', 248, 9, 'V9329', 244, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1985/'),
        1986: ('V5010', 256, 9, 'V9329', 252, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1986/'),
        1987: ('V5010', 248, 9, 'V9329', 244, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1987/'),
        1988: ('V5010', 248, 9, 'V9329', 244, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1988/'),
        1989: ('V5010', 60, 9, 'V9329', 245, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1989/'),
        1990: ('V5010', 56, 9, 'V9329', 244, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1990/PND1990N.DAT'),
        1991: (None, None, None, None, None, None, None),
        1992: ('V4614', 139, 12, 'V0105', 15, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1992/'),
        1993: ('V4614', 139, 12, 'V0105', 15, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1993/'),
        1994: (None, None, None, None, None, None, None),
        1995: ('V4614', 139, 12, 'V0105', 15, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1995/'),
        1996: ('V4614', 139, 12, 'V0105', 15, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1996/'),
        1997: ('V4614', 139, 12, 'V0105', 15, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1997/'),
        1998: ('V4614', 142, 12, 'V0105', 15, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1998/'),
        1999: ('V4614', 142, 12, 'V0105', 15, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1999/'),
        2000: (None, None, None, None, None, None, None),
        2001: ('V4614', 146, 12, 'V0105', 17, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2001.zip'),
        2002: ('V4614', 153, 12, 'V0105', 17, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2002.zip'),
        2003: ('V4614', 153, 12, 'V0105', 17, 2, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2003_20150814.zip'),
        2004: ('V4621', 239, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2004.zip'),
        2005: ('V4621', 181, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2005.zip'),
        2006: ('V4621', 181, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2006.zip'),
        2007: ('V4621', 179, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2007_20150814.zip'),
        2008: ('V4621', 181, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2008.zip'),
        2009: ('V4621', 181, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2009_20171228.zip'),
        2010: (None, None, None, None, None, None, None),
        2011: ('V4621', 176, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2011_20150814.zip'),
        2012: ('V4621', 176, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/reponderacao_2001_2012/PNAD_reponderado_2012_20150814.zip'),
        2013: ('V4621', 193, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/2013/Dados_20170807.zip'),
        2014: ('V4621', 193, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/2014/Dados_20170323.zip'),
        2015: ('V4621', 193, 12, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/2015/Dados_20170517.zip'),
        2016: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2016/'),
        2017: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2017/'),
        2018: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2018/'),
        2019: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2019/'),
        2020: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2020/'),
        2021: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2021/'),
        2022: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2022/'),
        2023: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2023/'),
        2024: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2024/'),
        2025: ('VD4019', 443, 8, None, None, None, 'https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2025/')
    }  
    # define nomes das colunas
    cols = ['var_renda','pos_renda','tam_renda','var_morador','pos_morador','tam_morador','link']  
    # cria dataframe a partir do dicionário
    df = pd.DataFrame.from_dict(specs_pnad, orient='index', columns=cols) 
    # transforma índice em coluna ano
    df = df.reset_index().rename(columns={'index':'ano'})  
    # especificações de ingestão dos arquivos brutos
    available = df['pos_renda'].notna()

    df['raw_subdir'] = ''
    df.loc[df['ano'].isin([1983, 1988]), 'raw_subdir'] = df.loc[
        df['ano'].isin([1983, 1988]), 'ano'
    ].astype(str)

    df['raw_pattern'] = ''
    df.loc[available, 'raw_pattern'] = (
        'DOM' + df.loc[available, 'ano'].astype(str) + '.*'
    )
    df.loc[df['ano'] == 1983, 'raw_pattern'] = 'PND83RM*.DAT'
    df.loc[df['ano'] == 1988, 'raw_pattern'] = 'PND88RM*.DAT'

    df['n_files'] = 0
    df.loc[available, 'n_files'] = 1
    df.loc[df['ano'].isin([1983, 1988]), 'n_files'] = 8

    df['missing_renda'] = np.nan
    df.loc[df['ano'].between(1977, 1990) & available, 'missing_renda'] = 999_999_999
    df.loc[df['ano'].between(1992, 2015) & available, 'missing_renda'] = 999_999_999_999
    df.loc[df['ano'].between(2016, 2025) & available, 'missing_renda'] = 99_999_999
    df.loc[df['ano'].isin([1976, 1981, 1982, 1983, 1984]), 'missing_renda'] = 9_999_999

    # campos textuais vazios para anos sem pesquisa
    df['link'] = df['link'].fillna('')
    df['raw_subdir'] = df['raw_subdir'].fillna('')
    df['raw_pattern'] = df['raw_pattern'].fillna('')

    # ordena por ano e reseta índice
    df = df.sort_values('ano').reset_index(drop=True)
    return df

# executa função
df_specs = build_specs_pnad_df()

# valida a especificação de ingestão
available = df_specs['pos_renda'].notna()
assert (df_specs.loc[available, 'raw_pattern'] != '').all()
assert (df_specs.loc[available, 'n_files'] > 0).all()
assert df_specs.loc[available, 'missing_renda'].notna().all()

# exibe resultado
df_specs


,ano,var_renda,pos_renda,tam_renda,var_morador,pos_morador,tam_morador,link,raw_subdir,raw_pattern,n_files,missing_renda
0,1976,V2954,227.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1976/,,DOM1976.*,1,9.999999e+06
1,1977,V131,288.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1977/,,DOM1977.*,1,1.000000e+09
2,1978,V2541,214.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1978/,,DOM1978.*,1,1.000000e+09
3,1979,V2517,167.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1979/,,DOM1979.*,1,1.000000e+09
4,1980,NaN,NaN,NaN,NaN,NaN,NaN,,,,0,NaN
5,1981,V5010,223.0,7.0,V9329,219.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1981/,,DOM1981.*,1,9.999999e+06
6,1982,V602,199.0,7.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1982/,,DOM1982.*,1,9.999999e+06
7,1983,V5010,223.0,7.0,V9329,219.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1983/,1983,PND83RM*.DAT,8,9.999999e+06
8,1984,V5010,223.0,7.0,V9329,219.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1984/,,DOM1984.*,1,9.999999e+06
9,1985,V5010,248.0,9.0,V9329,244.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1985/,,DOM1985.*,1,1.000000e+09


# Data Normalization and Economic Adjustment Framework

This project constructs a consistent longitudinal dataset of household income in Brazil by integrating PNAD microdata across heterogeneous survey structures and time periods. A critical step in this process is the normalization of monetary values to ensure **temporal comparability** under varying currency regimes, inflation dynamics, and exchange rate fluctuations. To achieve this, two macroeconomic adjustment factors are incorporated:

1. **Exchange Rate (BRL/USD)** — used to normalize income values into a common currency reference.
2. **Inflation Index (CPI)** — used to express all monetary values in real terms relative to a base year (2025).

The adjustment is defined as:

- **Adjust2025** = $\frac{\text{Index}_{2025}}{\text{Index}_{t}} - 1$
- **Inflation** = $1 + \text{Adjust2025} = \frac{\text{Index}_{2025}}{\text{Index}_{t}}$

After conversion to USD using the period-average exchange rate, this formulation rescales historical income values to **constant 2025 US-dollar purchasing power (CPI-U basis)**, enabling robust cross-temporal statistical analysis.

## Data Sources

- **Exchange Rate Source (Brazilian Central Bank — BCB)**  
  https://www3.bcb.gov.br/sgspub/consultarvalores/consultarValoresSeries.do?method=consultarValores  
  **Series:** 3698 — *Exchange rate - Free - United States dollar (sale) - period average*

- **Inflation Index Source (Federal Reserve Economic Data — FRED)**  
  https://fred.stlouisfed.org/series/CPIAUCSL  
  **Series:** CPIAUCSL — *Consumer Price Index for All Urban Consumers (CPI-U), All Items*

The resulting dataset provides:

- Income values normalized to a **single currency baseline**
- Adjustment to **constant 2025 US-dollar purchasing power (CPI-U basis)**
- Compatibility with **econophysics and distributional analysis frameworks**

This ensures that observed dynamics in income distributions reflect **structural economic behavior**, rather than nominal distortions.

In [3]:
df_currency = pd.DataFrame({
    "Year": list(range(1976, 2026)),
    "Currency": [
        "cruzeiro-Cr$", "cruzeiro-Cr$", "cruzeiro-Cr$", "cruzeiro-Cr$", "cruzeiro-Cr$",
        "cruzeiro-Cr$", "cruzeiro-Cr$", "cruzeiro-Cr$", "cruzeiro-Cr$", "cruzeiro-Cr$",
        "cruzado-Cz$", "cruzado-Cz$", "cruzado-Cz$", "cruzado novo-NCz$", "cruzeiro-Cr$",
        "cruzeiro-Cr$", "cruzeiro-Cr$", "cruzeiro real-CR$", "Real R$", "Real R$",
        "Real R$", "Real R$", "Real R$", "Real R$", "Real R$",
        "Real R$", "Real R$", "Real R$", "Real R$", "Real R$",
        "Real R$", "Real R$", "Real R$", "Real R$", "Real R$",
        "Real R$", "Real R$", "Real R$", "Real R$", "Real R$",
        "Real R$", "Real R$", "Real R$", "Real R$", "Real R$",
        "Real R$", "Real R$", "Real R$", "Real R$", "Real R$"
    ],
    "Exchange": [
        11.3130, 14.9300, 19.0500, 28.7920, np.nan,
        105.2840, 202.0880, 701.3810, 2203.9470, 7461.6670,
        13.8400, 49.8660, 326.2350, 3.2670, 75.5410,
        np.nan, 5771.5240, 111.1890, np.nan, 0.9528,
        1.0193, 1.0936, 1.1809, 1.8981, np.nan,
        2.6717, 3.3420, 2.9228, 2.8911, 2.2944,
        2.1687, 1.8996, 1.7996, 1.8198, np.nan,
        1.7498, 2.0281, 2.2705, 2.3329, 3.9065,
        3.2564, 3.1348, 4.1165, 4.1215, 5.3995,
        5.2797, 5.2370, 4.9370, 5.5416, 5.3674
    ],
    "Index": [
        100.00000, 106.42361, 115.45139, 129.16667, np.nan,
        161.63194, 169.61806, 174.30556, 181.77083, 187.67361,
        190.97222, 199.13194, 207.46528, 216.66667, 230.03472,
        np.nan, 244.96528, 251.73611, np.nan, 265.79861,
        273.78472, 279.86111, 283.85417, 291.31944, np.nan,
        309.20139, 313.88889, 321.35417, 329.51389, 345.13889,
        352.08333, 362.06076, 379.99479, 374.75868, np.nan,
        393.39757, 401.06771, 405.45833, 412.28646, 412.32292,
        418.70833, 427.83854, 437.81597, 445.19097, 451.38368,
        475.53819, 514.49479, 533.46528, 546.40972, 562.92535
    ]
})

idx_2025 = df_currency.loc[df_currency["Year"] == 2025, "Index"].iloc[0]
mask = df_currency["Index"].notna()
df_currency.loc[mask, "Adjust2025"] = (idx_2025 / df_currency.loc[mask, "Index"]) - 1
df_currency.loc[mask, "Inflation"] = df_currency.loc[mask, "Adjust2025"] + 1
df_currency

,Year,Currency,Exchange,Index,Adjust2025,Inflation
0,1976,cruzeiro-Cr$,11.3130,100.00000,4.629253,5.629253
1,1977,cruzeiro-Cr$,14.9300,106.42361,4.289478,5.289478
2,1978,cruzeiro-Cr$,19.0500,115.45139,3.875865,4.875865
3,1979,cruzeiro-Cr$,28.7920,129.16667,3.358132,4.358132
4,1980,cruzeiro-Cr$,NaN,NaN,NaN,NaN
5,1981,cruzeiro-Cr$,105.2840,161.63194,2.482761,3.482761
6,1982,cruzeiro-Cr$,202.0880,169.61806,2.318782,3.318782
7,1983,cruzeiro-Cr$,701.3810,174.30556,2.229532,3.229532
8,1984,cruzeiro-Cr$,2203.9470,181.77083,2.096896,3.096896
9,1985,cruzeiro-Cr$,7461.6670,187.67361,1.999491,2.999491


# Cria e salva o df_metadata


In [4]:
METADATA_PATH = Path(
    r"C:\Users\Osvaldo\OneDrive\academic_research\econophysics\projeto_pnad_ic_beatriz\metadata\df_metadata.xlsx"
)

if not METADATA_PATH.parent.is_dir():
    raise FileNotFoundError(
        f"Pasta de metadata não encontrada: {METADATA_PATH.parent}"
    )

df_metadata = (
    df_specs
    .merge(df_currency, left_on="ano", right_on="Year", how="left")
    .drop(columns="Year")
)

assert df_metadata["ano"].is_unique
assert df_metadata["ano"].tolist() == list(range(1976, 2026))

df_metadata.to_excel(METADATA_PATH, index=False)

df_metadata


,ano,var_renda,pos_renda,tam_renda,var_morador,pos_morador,tam_morador,link,raw_subdir,raw_pattern,n_files,missing_renda,Currency,Exchange,Index,Adjust2025,Inflation
0,1976,V2954,227.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1976/,,DOM1976.*,1,9.999999e+06,cruzeiro-Cr$,11.3130,100.00000,4.629253,5.629253
1,1977,V131,288.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1977/,,DOM1977.*,1,1.000000e+09,cruzeiro-Cr$,14.9300,106.42361,4.289478,5.289478
2,1978,V2541,214.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1978/,,DOM1978.*,1,1.000000e+09,cruzeiro-Cr$,19.0500,115.45139,3.875865,4.875865
3,1979,V2517,167.0,9.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1979/,,DOM1979.*,1,1.000000e+09,cruzeiro-Cr$,28.7920,129.16667,3.358132,4.358132
4,1980,NaN,NaN,NaN,NaN,NaN,NaN,,,,0,NaN,cruzeiro-Cr$,NaN,NaN,NaN,NaN
5,1981,V5010,223.0,7.0,V9329,219.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1981/,,DOM1981.*,1,9.999999e+06,cruzeiro-Cr$,105.2840,161.63194,2.482761,3.482761
6,1982,V602,199.0,7.0,NaN,NaN,NaN,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1982/,,DOM1982.*,1,9.999999e+06,cruzeiro-Cr$,202.0880,169.61806,2.318782,3.318782
7,1983,V5010,223.0,7.0,V9329,219.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1983/,1983,PND83RM*.DAT,8,9.999999e+06,cruzeiro-Cr$,701.3810,174.30556,2.229532,3.229532
8,1984,V5010,223.0,7.0,V9329,219.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1984/,,DOM1984.*,1,9.999999e+06,cruzeiro-Cr$,2203.9470,181.77083,2.096896,3.096896
9,1985,V5010,248.0,9.0,V9329,244.0,2.0,https://ftp.ibge.gov.br/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_anual/microdados/1985/,,DOM1985.*,1,1.000000e+09,cruzeiro-Cr$,7461.6670,187.67361,1.999491,2.999491
